In [67]:
import numpy as np
import pandas as pd

Load raw_trades.csv, then report its shape, dtypes and null counts in three lines. 

In [68]:
# path is relative to this notebook, so it works on any clone of the repo
raw_trades = pd.read_csv('../data/raw_trades.csv')

In [69]:
raw_trades.head()

,trade_id,ticker,trade_date,price,volume,venue_code,side
0,1044,JNJ,2026-01-03,NaN,1000.0,XNYS,SELL
1,1045,JNJ,2026-01-03,NaN,7500.0,XNYS,SELL
2,1012,MSFT,2026-01-07,438.92,1500.0,XNAS,SELL
3,1002,AAPL,2026-01-08,224.96,1000.0,XNAS,BUY
4,1013,MSFT,2026-01-11,447.60,7500.0,XNAS,BUY


In [70]:
#show the shape of the dataframe (rows, columns)
raw_trades.shape

(56, 7)

In [71]:
#inspect the different data types
raw_trades.dtypes

trade_id        int64
ticker            str
trade_date        str
price         float64
volume        float64
venue_code        str
side              str
dtype: object

In [72]:
#count how many rows are null per column
raw_trades.isna().sum()

trade_id      0
ticker        0
trade_date    0
price         6
volume        3
venue_code    0
side          0
dtype: int64

Set trade_date as the index and select January's trades two ways — with .loc and with .iloc — and write one line on when each is appropriate.

In [73]:
#set trade date as index
raw_trades.set_index("trade_date", inplace=True)

In [74]:
raw_trades.head()

,trade_id,ticker,price,volume,venue_code,side
trade_date,,,,,,
2026-01-03,1044,JNJ,NaN,1000.0,XNYS,SELL
2026-01-03,1045,JNJ,NaN,7500.0,XNYS,SELL
2026-01-07,1012,MSFT,438.92,1500.0,XNAS,SELL
2026-01-08,1002,AAPL,224.96,1000.0,XNAS,BUY
2026-01-11,1013,MSFT,447.60,7500.0,XNAS,BUY


In [75]:
raw_trades.sort_index(ascending=True, inplace=True)

In [76]:
raw_trades.head()

,trade_id,ticker,price,volume,venue_code,side
trade_date,,,,,,
2026-01-03,1044,JNJ,NaN,1000.0,XNYS,SELL
2026-01-03,1045,JNJ,NaN,7500.0,XNYS,SELL
2026-01-07,1012,MSFT,438.92,1500.0,XNAS,SELL
2026-01-08,1002,AAPL,224.96,1000.0,XNAS,BUY
2026-01-11,1013,MSFT,447.60,7500.0,XNAS,BUY


In [77]:
#select january trades
#start and end
raw_trades.loc['2026-01-01':'2026-01-31']

,trade_id,ticker,price,volume,venue_code,side
trade_date,,,,,,
2026-01-03,1044,JNJ,NaN,1000.0,XNYS,SELL
2026-01-03,1045,JNJ,NaN,7500.0,XNYS,SELL
2026-01-07,1012,MSFT,438.92,1500.0,XNAS,SELL
2026-01-08,1002,AAPL,224.96,1000.0,XNAS,BUY
2026-01-11,1013,MSFT,447.60,7500.0,XNAS,BUY
2026-01-13,1023,JPM,201.75,750.0,XNYS,BUY
2026-01-15,1001,AAPL,230.02,2500.0,XNAS,SELL
2026-01-17,1024,JPM,206.01,750.0,XNYS,SELL
2026-01-27,1034,XOM,124.22,NaN,XNYS,BUY


In [78]:
#positional
raw_trades.iloc[:9]

,trade_id,ticker,price,volume,venue_code,side
trade_date,,,,,,
2026-01-03,1044,JNJ,NaN,1000.0,XNYS,SELL
2026-01-03,1045,JNJ,NaN,7500.0,XNYS,SELL
2026-01-07,1012,MSFT,438.92,1500.0,XNAS,SELL
2026-01-08,1002,AAPL,224.96,1000.0,XNAS,BUY
2026-01-11,1013,MSFT,447.60,7500.0,XNAS,BUY
2026-01-13,1023,JPM,201.75,750.0,XNYS,BUY
2026-01-15,1001,AAPL,230.02,2500.0,XNAS,SELL
2026-01-17,1024,JPM,206.01,750.0,XNYS,SELL
2026-01-27,1034,XOM,124.22,NaN,XNYS,BUY


**`.loc` vs `.iloc`:** use `.loc` when you mean the data itself — a date range like January — because labels survive sorting, filtering and row drops; use `.iloc` when you genuinely mean position rather than content, such as taking the top 5 after a deliberate sort, and accept that those positions shift the moment the frame changes.

Which column has the wrong dtype straight off the CSV, and why did that happen?

**`trade_date` comes in as a string when it should be a datetime.** A CSV carries no type information at all — every value in the file is plain text. pandas infers types on read, and it can safely infer numbers, but dates are ambiguous (is `01/02/2026` 1 February or 2 January?), so it refuses to guess and leaves them as strings unless told explicitly via `parse_dates`. In BigQuery `trade_date` is declared `DATE` in the schema, so the type travels with the data; exporting to CSV throws that away.

**Also worth noting: `volume` reads back as `float64`, not an integer**, despite share volumes being whole numbers. `NaN` is a float value and NumPy's `int64` has no representation for "missing", so a single null forces the entire column to upcast to float. Once the nulls are handled on Tuesday this can be converted back to `int`, or to pandas' nullable `Int64` which holds missing values while staying integer.